# Downstream Task Application

## 0. Install Packages

In [ ]:
!pip install timm torchmetrics

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Path Configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/Project-IAS'
DOWNSTREAM_SRC = f'{PROJECT_ROOT}/downstream_task_src'

DATASET_PATH = '/content/drive/MyDrive/AD_dataset_application3'
DEFECT_TYPE = 'mounting_plate'

EPOCHS = 400
BATCH_SIZE = 16
TRAIN_CKPT_DIR = '/content/draem_train_output'
DRAEM_CKPT_DIR = f'{PROJECT_ROOT}/model/draem'
EVAL_OUTPUT_DIR = '/content/anomalydiffusion_result'

for path in [DOWNSTREAM_SRC, f'{DATASET_PATH}/{DEFECT_TYPE}']:
    if not Path(path).exists():
        raise FileNotFoundError(f'경로를 찾을 수 없습니다: {path}')

print(f'데이터셋 : {DATASET_PATH}')

## 3. Train DRAEM

In [ ]:
%cd $DOWNSTREAM_SRC

!python train_draem.py \
      --dataset_path $DATASET_PATH \
      --defect_types $DEFECT_TYPE \
      --epochs $EPOCHS \
      --bs $BATCH_SIZE \
      --checkpoint_path $TRAIN_CKPT_DIR

## 4. Copy Checkpoints to Drive

In [ ]:
LAST_EPOCH = EPOCHS - 1

!mkdir -p {DRAEM_CKPT_DIR}
!cp -r {TRAIN_CKPT_DIR}/draem_rec_{LAST_EPOCH}.pckl {TRAIN_CKPT_DIR}/draem_seg_{LAST_EPOCH}.pckl {DRAEM_CKPT_DIR}/

CKPT_REC = f'{DRAEM_CKPT_DIR}/draem_rec_{LAST_EPOCH}.pckl'
CKPT_SEG = f'{DRAEM_CKPT_DIR}/draem_seg_{LAST_EPOCH}.pckl'

print(f'reconstructive : {CKPT_REC}')
print(f'discriminative : {CKPT_SEG}')

## 5. Evaluate

In [ ]:
%cd $DOWNSTREAM_SRC

!python eval_draem.py \
      --dataset_path $DATASET_PATH \
      --defect_types $DEFECT_TYPE \
      --checkpoint_rec $CKPT_REC \
      --checkpoint_seg $CKPT_SEG \
      --checkpoint_path $EVAL_OUTPUT_DIR

## 6. Check Results

In [ ]:
import json

results = json.loads((Path(EVAL_OUTPUT_DIR) / 'eval_results.json').read_text())
for key, value in results.items():
    print(f'{key:>28} : {value}')